# 🚀 ZAC 2026 LifeMate — v7.0 Pipeline

**Cấu hình thư mục:**
- Đọc dữ liệu và các checkpoint ASR có sẵn từ thư mục dự án (`ZaloAI2026` trên Shared with me / Shortcut Drive hoặc thư mục máy local).
- Hỗ trợ cả định dạng **thư mục đã giải nén sẵn** (`lifemate_code/`, `train+public_test/`) lẫn file `.zip`.
- **Không đẩy kết quả mới lên Google Drive:** Toàn bộ **Checkpoint v7** (`qwen3_4b_sft_lora_v7`), **Eval** (`val_pred_v7_e2e.jsonl`, `eval_errors_v7_e2e.jsonl`), **Timing** (`timing_val_v7_e2e.jsonl`, `timing_public_v7.jsonl`) và **Test** (`submission_v7.jsonl`) đều được lưu trực tiếp trong thư mục project (`OUT_DIR`).

| Ô | Công việc | File lưu trong Project (`OUT_DIR`) |
|---|---|---|
| 1 | Kết nối thư mục dự án `ZaloAI2026` · kiểm tra GPU · cài đặt thư viện | `WORK_DIR`, `DATA_ROOT`, `OUT_DIR` |
| 2 | Kiểm tra Unit Tests Guardrails v7.0 + Oracle Check (99.94/100) | Kiểm tra logic Agent & ToolEnv |
| 3 | Kiểm tra / Fine-tune **PhoWhisper-medium** (bỏ qua nếu đã có checkpoint) | `asr_pho_ft_v1/` |
| 4 | Kiểm tra / Fine-tune **Whisper-large-v3-turbo** (bỏ qua nếu đã có checkpoint) | `asr_turbo_ft_v1/` |
| 5 | Xây dựng tập dữ liệu **SFT v7.0** (Nguồn A + Nguồn B + Recovery Training) | `sft_train_v7.jsonl` |
| 6 | Huấn luyện **Qwen3-4B PureLoRA v7.0** (`--free-text-weight 0.3`, lưu `epoch_1`, `epoch_2` & `--save-steps 150`) | `qwen3_4b_sft_lora_v7/` |
| 7 | **Đánh giá Validation End-to-End** (ASR thật + Step-Level Ensemble) & phân tích lỗi | `val_pred_v7_e2e.jsonl`, `eval_errors_v7_e2e.jsonl`, `timing_val_v7_e2e.jsonl` |
| 8 | Kiểm tra / Chạy ASR `public_test` bằng 2 model đã fine-tune | `public_test_asr_ft_v1_cache.json` |
| 9 | **Suy luận `public_test` v7.0** (Multi-ASR Hypotheses + Step-Level Ensemble + `--resume`) | `submission_v7.jsonl`, `timing_public_v7.jsonl` |
| 10 | Kiểm tra Schema bằng `zaic_tools` · Thống kê thời gian & so sánh với `v5`/`v6` | Báo cáo tổng kết |

In [ ]:
# === Ô 1: Kết nối thư mục dự án · Kiểm tra GPU · Cài đặt thư viện ===
import os, glob, shutil, zipfile, subprocess
from pathlib import Path

# 1) Tự động tìm thư mục dự án (trên máy local hoặc Shared with me / ZaloAI2026 trên Colab)
LOCAL_DIR = Path('/home/leatuyrbertyk/Documents/ZAIC26')
if LOCAL_DIR.is_dir():
    SRC_DIR  = str(LOCAL_DIR)
    WORK_DIR = f'{SRC_DIR}/lifemate_code'
    OUT_DIR  = SRC_DIR  # Lưu eval, checkpoint, timing, test ngay trong project
else:
    from google.colab import drive
    drive.mount('/content/drive')
    # Tìm ZaloAI2026 trong MyDrive, Shared with me (.shortcut-targets-by-id), hoặc Shareddrives
    candidates = (
        glob.glob('/content/drive/MyDrive/ZaloAI2026')
        + glob.glob('/content/drive/.shortcut-targets-by-id/*/ZaloAI2026')
        + glob.glob('/content/drive/Shareddrives/*/ZaloAI2026')
        + glob.glob('/content/drive/Shareddrives/ZaloAI2026')
    )
    if not candidates:
        raise FileNotFoundError(
            '❌ Không thấy ZaloAI2026! Với thư mục nằm ở "Shared with me / ZaloAI2026", '
            'bạn hãy chuột phải vào thư mục ZaloAI2026 trên Google Drive -> chọn "Add shortcut to Drive" (Thêm lối tắt vào Drive) -> chọn My Drive rồi chạy lại ô này.'
        )
    SRC_DIR  = candidates[0]
    WORK_DIR = '/content/lifemate_project'
    OUT_DIR  = '/content/lifemate_project/outputs_v7'  # Lưu cục bộ trong project, KHÔNG đẩy lên Drive
    os.makedirs(WORK_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)

    # Đồng bộ code v7.0 mới nhất từ GitHub repo (Public) vào WORK_DIR
    if not os.path.isdir('/content/zaic26_git/.git'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/LeatuyrBertyk/ZAIC26.git', '/content/zaic26_git'], check=False)
    else:
        subprocess.run(['git', '-C', '/content/zaic26_git', 'pull', '--ff-only'], check=False)
    if os.path.isdir('/content/zaic26_git/lifemate_code'):
        shutil.copytree('/content/zaic26_git/lifemate_code', WORK_DIR, dirs_exist_ok=True)
    elif os.path.isdir(f'{SRC_DIR}/lifemate_code'):
        shutil.copytree(f'{SRC_DIR}/lifemate_code', WORK_DIR, dirs_exist_ok=True)
    elif os.path.isfile(f'{SRC_DIR}/lifemate_code.zip'):
        with zipfile.ZipFile(f'{SRC_DIR}/lifemate_code.zip') as z:
            z.extractall(WORK_DIR)

# 2) Chuẩn bị DATA_ROOT (hỗ trợ cả thư mục train+public_test đã giải nén hoặc file .zip)
if os.path.isfile(f'{SRC_DIR}/train+public_test/train/train.jsonl'):
    DATA_ROOT = f'{SRC_DIR}/train+public_test'
elif os.path.isfile(f'{SRC_DIR}/train+public_test/train+public_test/train/train.jsonl'):
    DATA_ROOT = f'{SRC_DIR}/train+public_test/train+public_test'
else:
    RAW_DIR = '/content/data_raw'
    if not glob.glob(f'{RAW_DIR}/**/public_test/public_test.jsonl', recursive=True):
        os.makedirs(RAW_DIR, exist_ok=True)
        with zipfile.ZipFile(f'{SRC_DIR}/train+public_test.zip') as z:
            z.extractall(RAW_DIR)
    hit = glob.glob(f'{RAW_DIR}/**/public_test/public_test.jsonl', recursive=True)[0]
    DATA_ROOT = os.path.dirname(os.path.dirname(hit))

link_parent = f'{WORK_DIR}/train+public_test'
link_target = f'{link_parent}/train+public_test'
os.makedirs(link_parent, exist_ok=True)
if not os.path.lexists(link_target):
    os.symlink(DATA_ROOT, link_target)

def get_path(name: str) -> str:
    """Ưu tiên tìm file/checkpoint trong OUT_DIR trước, nếu chưa có thì lấy từ SRC_DIR."""
    p_out = f'{OUT_DIR}/{name}'
    if os.path.exists(p_out):
        return p_out
    p_src = f'{SRC_DIR}/{name}'
    return p_src if os.path.exists(p_src) else p_out

# 3) Kiểm tra GPU
ret = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = ret.stdout.strip() if ret.returncode == 0 else 'CPU'
SMALL_GPU = any(k in GPU_NAME for k in ('L4', 'T4', 'CPU'))
ASR_BS, ASR_GA = (4, 4) if SMALL_GPU else (8, 2)
TIME_BUDGET = 150 if SMALL_GPU else 25

print(f'✅ GPU      = {GPU_NAME} (SMALL_GPU={SMALL_GPU} | time-budget={TIME_BUDGET}s)')
print(f'📂 SRC_DIR  = {SRC_DIR}')
print(f'📂 WORK_DIR = {WORK_DIR}')
print(f'📂 DATA_ROOT= {DATA_ROOT}')
print(f'📂 OUT_DIR  = {OUT_DIR} (Nơi lưu eval, checkpoint, timing, test — KHÔNG đẩy lên Drive)')

!pip install -q -U faster-whisper "transformers>=4.51.0" accelerate jiwer soundfile librosa
os.chdir(WORK_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU      = Tesla T4, 15360 MiB (SMALL_GPU=True | time-budget=150s)
📂 SRC_DIR  = /content/drive/MyDrive/ZaloAI2026
📂 WORK_DIR = /content/lifemate_project
📂 DATA_ROOT= /content/data_raw/train+public_test
📂 OUT_DIR  = /content/lifemate_project/outputs_v7 (Nơi lưu eval, checkpoint, timing, test — KHÔNG đẩy lên Drive)


In [ ]:
# === Ô 2: Kiểm tra Unit Tests Guardrails v7.0 + Oracle Check (99.94/100) ===
import os
os.chdir(WORK_DIR)
if not os.path.isfile('outputs/splits/val_split.jsonl'):
    !python3 -X utf8 scripts/make_splits.py
!python3 -X utf8 scripts/test_agent_guardrails.py
!python3 -X utf8 scripts/oracle_agent_check.py --input outputs/splits/val_split.jsonl --show 5


PASS 1 sanitizer
PASS 2 ask_missing_info targets
PASS 3-5 schema / x-ref / duplicate-write guards -> ['smart_home.get_status', 'alarm.create']
PASS 4b cross-app x-ref accepted
PASS 6 out-of-scope -> []
PASS 7 context budget (history trimmed to 18 messages)
[Agent] train_0001: time budget reached, stopping.
PASS 8 time budget
PASS 9 vague amount -> ask_missing_info
ALL GUARDRAIL TESTS PASSED
Oracle-through-agent score: 99.60 / 100 on 250 samples
Damaged samples: 1 | by category: {'multi_step_write': 1}
{"id": "train_0922", "cat": "multi_step_write", "score": 0.0, "gold": [{"name": "loyalty.get_points", "arguments": {"program": "Cao Nguyên"}}, {"name": "loyalty.redeem", "arguments": {"program": "prog_815", "reward_id": "rw_02_1"}}, {"name": "loyalty.redeem", "arguments": {"program": "prog_815", "reward_id": "rw_02_1"}}], "pred": [{"name": "loyalty.get_points", "arguments": {"program": "Cao Nguyên"}}, {"name": "loyalty.redeem", "arguments": {"program": "prog_815", "reward_id": "rw_02_1"}}

In [ ]:
# === Ô 3: Kiểm tra / Fine-tune PhoWhisper-medium (0.77B) ===
import os
os.chdir(WORK_DIR)
PHO_OUT = get_path('asr_pho_ft_v1')
if os.path.isfile(f'{PHO_OUT}/asr_ft_report.json'):
    print('✅ Đã có checkpoint PhoWhisper hoàn chỉnh tại:', PHO_OUT)
    print(open(f'{PHO_OUT}/asr_ft_report.json', encoding='utf-8').read())
else:
    PHO_OUT = f'{OUT_DIR}/asr_pho_ft_v1'
    base_pho = PHO_OUT if os.path.isfile(f'{PHO_OUT}/config.json') else 'vinai/PhoWhisper-medium'
    !python3 -X utf8 scripts/finetune_asr.py --base-model "{base_pho}" --output-dir "{PHO_OUT}" \
      --data-root "{DATA_ROOT}" --epochs 3 --lr 1e-5 --batch-size {ASR_BS} --grad-accum {ASR_GA} --save-steps 250 --eval-baseline


✅ Đã có checkpoint PhoWhisper hoàn chỉnh tại: /content/drive/MyDrive/ZaloAI2026/asr_pho_ft_v1
{
  "base_model": "vinai/PhoWhisper-medium",
  "train_utterances": 4248,
  "heldout_utterances": 250,
  "wer_before": 0.2859598853868195,
  "wer_after": 0.11031518624641834,
  "steps": 798
}


In [ ]:
# === Ô 4: Kiểm tra / Fine-tune Whisper-large-v3-turbo (0.81B) ===
import os
os.chdir(WORK_DIR)
FREEZE_ENCODER = False
TURBO_OUT = get_path('asr_turbo_ft_v1')
if os.path.isfile(f'{TURBO_OUT}/asr_ft_report.json'):
    print('✅ Đã có checkpoint Whisper-turbo hoàn chỉnh tại:', TURBO_OUT)
    print(open(f'{TURBO_OUT}/asr_ft_report.json', encoding='utf-8').read())
else:
    TURBO_OUT = f'{OUT_DIR}/asr_turbo_ft_v1'
    extra = '--freeze-encoder' if FREEZE_ENCODER else ''
    base_turbo = TURBO_OUT if os.path.isfile(f'{TURBO_OUT}/config.json') else 'openai/whisper-large-v3-turbo'
    !python3 -X utf8 scripts/finetune_asr.py --base-model "{base_turbo}" --output-dir "{TURBO_OUT}" \
      --data-root "{DATA_ROOT}" --epochs 3 --lr 1e-5 --batch-size {ASR_BS} --grad-accum {ASR_GA} --save-steps 250 --eval-baseline {extra}


✅ Đã có checkpoint Whisper-turbo hoàn chỉnh tại: /content/drive/MyDrive/ZaloAI2026/asr_turbo_ft_v1
{
  "base_model": "openai/whisper-large-v3-turbo",
  "train_utterances": 4248,
  "heldout_utterances": 250,
  "wer_before": 0.3616045845272206,
  "wer_after": 0.11060171919770774,
  "steps": 798
}


In [ ]:
# === Ô 5: Xây dựng dữ liệu SFT v7.0 (Lưu tại OUT_DIR/sft_train_v7.jsonl — KHÔNG đẩy lên Drive) ===
import os, shutil, subprocess
if WORK_DIR.startswith('/content'):
    if not os.path.isdir('/content/zaic26_git/.git'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/LeatuyrBertyk/ZAIC26.git', '/content/zaic26_git'], check=False)
    else:
        subprocess.run(['git', '-C', '/content/zaic26_git', 'pull', '--ff-only'], check=False)
    if os.path.isdir('/content/zaic26_git/lifemate_code'):
        shutil.copytree('/content/zaic26_git/lifemate_code', WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
SFT_V7_PATH = get_path('sft_train_v7.jsonl')
if os.path.isfile(SFT_V7_PATH):
    num_lines = sum(1 for _ in open(SFT_V7_PATH, encoding='utf-8'))
    print(f'✅ Đã có sẵn {SFT_V7_PATH} ({num_lines} mẫu huấn luyện).')
else:
    SFT_V7_PATH = f'{OUT_DIR}/sft_train_v7.jsonl'
    !python3 -X utf8 scripts/gen_data_source_a.py --input "{DATA_ROOT}/train/train.jsonl" --out outputs/sft/source_a_history.jsonl
    !python3 -X utf8 scripts/gen_data_source_b.py --num-samples 600 --out outputs/sft/source_b_simulator.jsonl
    val_cache = get_path('asr_val_ft_cache.json')
    asr_cache_arg = f'--asr-cache "{val_cache}"' if os.path.isfile(val_cache) else ''
    !python3 -X utf8 scripts/build_sft_data.py \
      --input-jsonl "{DATA_ROOT}/train/train.jsonl" \
      --exclude-val-jsonl outputs/splits/val_split.jsonl \
      --include-all-sources --synth-repeat 8 --recovery-count 240 {asr_cache_arg} \
      --out "{SFT_V7_PATH}"
    print('✅ Đã lưu tập SFT v7.0 tại:', SFT_V7_PATH)


In [ ]:
# === Ô 6: Huấn luyện Qwen3-4B PureLoRA v7.0 (Loss Weighting 0.3 + Checkpoint mỗi 150 bước & mỗi Epoch) ===
# Lưu trực tiếp vào OUT_DIR/qwen3_4b_sft_lora_v7 (kèm epoch_1/ và epoch_2/, KHÔNG đẩy lên Drive).
import os, shutil, subprocess
if WORK_DIR.startswith('/content'):
    if not os.path.isdir('/content/zaic26_git/.git'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/LeatuyrBertyk/ZAIC26.git', '/content/zaic26_git'], check=False)
    else:
        subprocess.run(['git', '-C', '/content/zaic26_git', 'pull', '--ff-only'], check=False)
    if os.path.isdir('/content/zaic26_git/lifemate_code'):
        shutil.copytree('/content/zaic26_git/lifemate_code', WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
LORA_V7_OUT = f'{OUT_DIR}/qwen3_4b_sft_lora_v7'
SFT_V7_PATH = get_path('sft_train_v7.jsonl')

if os.path.isfile(f'{LORA_V7_OUT}/epoch_2/pure_lora_weights.pt'):
    print('✅ Đã hoàn tất huấn luyện đủ 2 epochs tại:', LORA_V7_OUT)
else:
    resume_arg = f'--resume-from-adapter "{LORA_V7_OUT}"' if os.path.isfile(f'{LORA_V7_OUT}/pure_lora_weights.pt') else ''
    !python3 -X utf8 scripts/train_sft_lora.py \
      --base-model Qwen/Qwen3-4B-Instruct-2507 \
      --train-jsonl "{SFT_V7_PATH}" \
      --output-dir "{LORA_V7_OUT}" \
      --epochs 2.0 --lr 1.5e-4 --batch-size 1 --grad-accum 8 \
      --lora-r 32 --lora-alpha 64 --free-text-weight 0.3 --save-steps 150 {resume_arg}


In [ ]:
# === Ô 7: Đánh giá Validation END-TO-END v7.0 (ASR thật + Step-Level Majority Voting Ensemble) ===
# Lưu val_pred_v7_e2e.jsonl, eval_errors_v7_e2e.jsonl và timing_val_v7_e2e.jsonl trực tiếp trong OUT_DIR.
import os, shutil, subprocess, torch
if WORK_DIR.startswith('/content'):
    if not os.path.isdir('/content/zaic26_git/.git'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/LeatuyrBertyk/ZAIC26.git', '/content/zaic26_git'], check=False)
    else:
        subprocess.run(['git', '-C', '/content/zaic26_git', 'pull', '--ff-only'], check=False)
    if os.path.isdir('/content/zaic26_git/lifemate_code'):
        shutil.copytree('/content/zaic26_git/lifemate_code', WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
LORA_V7 = f'{OUT_DIR}/qwen3_4b_sft_lora_v7'
LORA_V5 = get_path('qwen3_4b_sft_lora_v5')
PRIMARY_LORA = LORA_V7 if os.path.isfile(f'{LORA_V7}/pure_lora_weights.pt') else LORA_V5
ENS_LIST = [p for p in [f'{LORA_V7}/epoch_1', LORA_V5] if os.path.isfile(f'{p}/pure_lora_weights.pt') and p != PRIMARY_LORA]
ENS_ARG = f'--llm-ensemble "{",".join(ENS_LIST)}"' if ENS_LIST else ''

PHO_OUT       = get_path('asr_pho_ft_v1')
TURBO_OUT     = get_path('asr_turbo_ft_v1')
VAL_ASR_CACHE = get_path('asr_val_ft_cache.json')
VAL_PRED      = f'{OUT_DIR}/val_pred_v7_e2e.jsonl'
VAL_ERRORS    = f'{OUT_DIR}/eval_errors_v7_e2e.jsonl'
VAL_TIMING    = f'{OUT_DIR}/timing_val_v7_e2e.jsonl'
MODEL_TAG     = f'{OUT_DIR}/.val_pred_model_tag'
if os.path.isfile(VAL_PRED) and (not os.path.isfile(MODEL_TAG) or open(MODEL_TAG).read().strip() != PRIMARY_LORA):
    os.remove(VAL_PRED)
    if os.path.isfile(VAL_TIMING):
        os.remove(VAL_TIMING)
open(MODEL_TAG, 'w').write(PRIMARY_LORA)

DTYPE = 'bfloat16' if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else 'float16'
print('Primary LoRA:', PRIMARY_LORA, '| Step-Ensemble:', ENS_LIST)

!python3 -X utf8 scripts/infer.py --data-root "{DATA_ROOT}" --split train \
  --input-jsonl outputs/splits/val_split.jsonl \
  --asr-mode transformers --asr-model "{TURBO_OUT}" --asr-secondary "{PHO_OUT}" \
  --asr-cache "{VAL_ASR_CACHE}" \
  --llm-backend transformers --llm-model "{PRIMARY_LORA}" {ENS_ARG} --dtype {DTYPE} \
  --time-budget {TIME_BUDGET} --out "{VAL_PRED}" --timing-out "{VAL_TIMING}" --resume

!python3 -X utf8 scripts/evaluate_local.py --sub "{VAL_PRED}" --only-present --error-out "{VAL_ERRORS}"


In [ ]:
# === Ô 8: Kiểm tra / Chạy ASR public_test bằng 2 model đã fine-tune ===
import os, time
os.chdir(WORK_DIR)
PHO_OUT      = get_path('asr_pho_ft_v1')
TURBO_OUT    = get_path('asr_turbo_ft_v1')
ASR_CACHE_FT = get_path('public_test_asr_ft_v1_cache.json')

t0 = time.perf_counter()
!python3 -X utf8 scripts/infer.py --data-root "{DATA_ROOT}" --split public_test \
  --asr-mode transformers --asr-model "{TURBO_OUT}" --asr-secondary "{PHO_OUT}" \
  --asr-cache "{ASR_CACHE_FT}" --asr-only
print(f'⏱ Hoàn tất kiểm tra/chạy ASR public_test trong {time.perf_counter()-t0:.1f}s -> {ASR_CACHE_FT}')


In [ ]:
# === Ô 9: Suy luận public_test v7.0 (Multi-ASR Hypotheses + Step-Level Ensemble + --resume) ===
# Lưu trực tiếp submission_v7.jsonl và timing_public_v7.jsonl trong OUT_DIR (KHÔNG đẩy lên Drive).
import os, torch
os.chdir(WORK_DIR)
LORA_V7 = f'{OUT_DIR}/qwen3_4b_sft_lora_v7'
LORA_V5 = get_path('qwen3_4b_sft_lora_v5')
PRIMARY_LORA = LORA_V7 if os.path.isfile(f'{LORA_V7}/pure_lora_weights.pt') else LORA_V5
ENS_LIST = [p for p in [f'{LORA_V7}/epoch_1', LORA_V5] if os.path.isfile(f'{p}/pure_lora_weights.pt') and p != PRIMARY_LORA]
ENS_ARG = f'--llm-ensemble "{",".join(ENS_LIST)}"' if ENS_LIST else ''

PHO_OUT        = get_path('asr_pho_ft_v1')
TURBO_OUT      = get_path('asr_turbo_ft_v1')
ASR_CACHE_FT   = get_path('public_test_asr_ft_v1_cache.json')
ASR_CACHE_BASE = get_path('public_test_whisper_turbo_cache.json')
BASE_ASR_ARG   = f'--asr-base-cache "{ASR_CACHE_BASE}"' if os.path.isfile(ASR_CACHE_BASE) else ''

DTYPE = 'bfloat16' if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else 'float16'
SUBMISSION = f'{OUT_DIR}/submission_v7.jsonl'
TIMING     = f'{OUT_DIR}/timing_public_v7.jsonl'

print('Primary LoRA:', PRIMARY_LORA, '| Step-Ensemble:', ENS_LIST)
!python3 -X utf8 scripts/infer.py --data-root "{DATA_ROOT}" --split public_test \
  --asr-mode transformers --asr-model "{TURBO_OUT}" --asr-secondary "{PHO_OUT}" \
  --asr-cache "{ASR_CACHE_FT}" {BASE_ASR_ARG} \
  --llm-backend transformers --llm-model "{PRIMARY_LORA}" {ENS_ARG} --dtype {DTYPE} \
  --time-budget {TIME_BUDGET} --out "{SUBMISSION}" --timing-out "{TIMING}" --resume


In [ ]:
# === Ô 10: Kiểm tra hợp lệ Schema · Thống kê Timing & So sánh với submission_v5 / submission_v6 ===
import os, json
SUBMISSION = f'{OUT_DIR}/submission_v7.jsonl'
TIMING     = f'{OUT_DIR}/timing_public_v7.jsonl'

for tag, path in [
    ('v5', get_path('submission_v5.jsonl')),
    ('v6', get_path('submission_v6.jsonl')),
    ('v7', SUBMISSION),
]:
    if os.path.isfile(path):
        rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
        empty = sum(not r['tool_calls'] for r in rows)
        total_calls = sum(len(r['tool_calls']) for r in rows)
        print(f'📊 [{tag}] Số dòng: {len(rows)} | Tổng calls: {total_calls} ({total_calls/len(rows):.2f} calls/mẫu) | Nộp [] : {empty} ({100*empty/len(rows):.1f}%)')

!PYTHONPATH="{DATA_ROOT}/public_test" python3 -X utf8 -m zaic_tools --split public_test --root "{DATA_ROOT}" --validate "{SUBMISSION}"

if os.path.isfile(TIMING):
    sec = sorted(json.loads(l)['seconds'] for l in open(TIMING, encoding='utf-8') if l.strip())
    n = len(sec)
    print(f'⏱ Thời gian / mẫu (v7): TB {sum(sec)/n:.2f}s | p50 {sec[n//2]:.2f}s | p95 {sec[int(.95*n)]:.2f}s | max {sec[-1]:.2f}s | >25s: {sum(s>25 for s in sec)}')
print(f'✅ Tất cả file kết quả nằm tại: {OUT_DIR}')
